<a href="https://colab.research.google.com/github/Protasovkr/quest5/blob/main/TimeSeriesAnalysis_%D0%B8%D1%82%D0%BE%D0%B3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Работа с временнЫми последовательностями

## Введение

- Объекты приходят последовательно во времени
- Хотим предсказывать какую-то информацию об объектах, которая будет в далёком будущем
- Фиксированный (обычно во времени) порядок всех объектов (отличие от обычной задачи предсказания)
- Может быть как задачей регрессии, так и задачей классификации
- В данной лекции будет рассматриваться только случай регрессии. Переход к классификации не должен составить труда

$y_0, y_1, y_2, ..., y_n$ &#151; временной ряд, $y_i \in \mathbb{R}$ &#151; значение, полученное через **постоянный** временной интервал.

$y_{T+d} \approx f_T(y_0, y_1, ..., y_T, \theta, d), d = 1, ..., D$ &#151; модель временного ряда,

где $\theta$ &#151; параметры модели, $D$ &#151; горизонт прогнозирования.

При $K = 1$ хотим предсказывать только одно следующее значение, в противном случае предсказываем упорядоченный набор $y_{t+1}, y_{t+2}, ..., y_{t+K}$.

В классической задаче анализа данных мы предполагаем независимость всех наблюдений. В случае анализа временных рядов мы исходим из гипотезы о том, что предсказываемое значение зависит от предыдущих.

## Пример задача прогнозирования:

1. курс валюты;
2. стоймость акций компании "Яндекс";
3. спрос на определённый продукт;
4. количество студентов без долгов в определенный момент времени;
5. процент посещаемости лекций по мат. анализу;
6. уровень безработицы;
7. ...

## Термины из области анализа временных рядов:

1. тренд;
2. сезонность, цикл;
3. ошибка.

In [11]:
from IPython.display import Image

### Тренд &#151; изменение значений ряда в долгосрочной преспективе

In [12]:
Image('./example_1.png')

FileNotFoundError: No such file or directory: './example_1.png'

FileNotFoundError: No such file or directory: './example_1.png'

<IPython.core.display.Image object>

In [13]:
Image('./example_2.png')

FileNotFoundError: No such file or directory: './example_2.png'

FileNotFoundError: No such file or directory: './example_2.png'

<IPython.core.display.Image object>

In [14]:
Image('./example_3.png')

FileNotFoundError: No such file or directory: './example_3.png'

FileNotFoundError: No such file or directory: './example_3.png'

<IPython.core.display.Image object>

### Сезонность &#151; циклические изменения значений ряда с *определённым* периодом

### Цикл &#151; циклические изменения значений ряда с *изменяемым* периодом

In [15]:
Image('./example_4.png')

FileNotFoundError: No such file or directory: './example_4.png'

FileNotFoundError: No such file or directory: './example_4.png'

<IPython.core.display.Image object>

In [16]:
Image('./example_5.png')

FileNotFoundError: No such file or directory: './example_5.png'

FileNotFoundError: No such file or directory: './example_5.png'

<IPython.core.display.Image object>

In [17]:
Image('./example_6.png')

FileNotFoundError: No such file or directory: './example_6.png'

FileNotFoundError: No such file or directory: './example_6.png'

<IPython.core.display.Image object>

### Ошибка &#151; непрогнозируеммая величина, шум.

In [18]:
Image('./example_7.png')

FileNotFoundError: No such file or directory: './example_7.png'

FileNotFoundError: No such file or directory: './example_7.png'

<IPython.core.display.Image object>

## Пример временного ряда:

In [19]:
# Импорт необходимых библиотек
import copy
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dateutil.relativedelta import relativedelta

# Source: https://www.statsmodels.org/stable/install.html
import statsmodels
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf, adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Source: https://scikit-learn.org/stable/install.html
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

%matplotlib inline

In [20]:
#df = pd.read_csv('./train.csv')
df = pd.read_csv('./example_data.csv')
df.head(13)

FileNotFoundError: [Errno 2] No such file or directory: './example_data.csv'

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# Проанализируем базовые статистики имеющихся признаков
df.describe()

In [ ]:
df.replace(np.nan, 0, inplace=True)

In [ ]:
# Привидём время к типу 'datetime' и сделаем его индексом для датасета
df['datetime'] = pd.to_datetime(df['datetime'], format='%m-%d-%Y') #%d.%m.%Y %H:%M:%S

df = df.set_index(['datetime'])
df.sort_index(inplace=True)
df.head()

In [ ]:
# Рассмотрим изменение цены
plt.figure(figsize=(14, 6))
plt.title('Time series example')
plt.xlabel('Date', fontsize=15)
plt.ylabel('total', rotation=0, labelpad=30, fontsize=15)
df['total'].plot();

Вопросы:
- Есть ли у данного ряда тренд?
- Есть ли у данного ряда сезонность?

In [ ]:
# Разложим ряд на компоненты и проверим ответы на вопросы

fig, ax = plt.subplots(nrows=4, ncols=1, figsize=(14, 40))
decompose = seasonal_decompose(df[['total']], period=90)

ax[0].set_title('Original')
ax[1].set_title('Trend')
ax[2].set_title('Seasonal')
ax[3].set_title('Residual')

decompose.observed.plot(ax=ax[0])
decompose.trend.plot(ax=ax[1])
decompose.seasonal.plot(ax=ax[2])
decompose.resid.plot(ax=ax[3]);

Для последующего анализа нам потребуется ввести термин "Автокорреляция", который позволит нам более подробно изучить свойства временных рядов

## Автокорреляция

### Автокорреляция &#151; мера силы линейной связи между значениями ряда в настоящем или прошлом

###  $$r_t = \frac{\mathbb{E}((y_t - \mathbb{E}y)(y_{t + \tau} - \mathbb{E}y))}{\mathbb{D}y}$$
$\tau$ &#151; лаг автокорреляции

$r_t \in [-1; 1]$

In [ ]:
# Получил 'сырые' значения автокорреляций
acf(df['total'], nlags=60, fft=False)

На самом же деле, с автокорреляциями удобнее работать имея перед глазами график, поэтому далее нам необходимо рассмотреть коррелограммы

### Корелограммы

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(15, 15))
df['total'].plot(ax=ax[0])
plot_acf(df['total'], lags=60, ax=ax[1]);

На втором графике каждый столбец -- значение автокорреляции, а зона, обозначенная синим, говорит о том, является ли определённая автокорреляция значимой (выходит из зоны) или нет (весь столбец находится в закрашенной области)

## Стационарность

Ряд $y_1, y_2, \cdots, y_T$ является **стационарным**, если для любого окна ширины $s$ распределение ряда $y_t, \cdots, y_{t+s}$ не зависит от выбора $t$

Некоторые полезные хинты для проверки ряда на стационарность:

- если у ряда есть тренд, то он нестационарен;
- если у ряда есть сезонность, то он нестационарен;
- если у ряда изменяется со временем дисперсия, то он нестационарен.

Есть ли критерий для проверки ряда на стационарность?

Ответ: Можно воспользоваться аппаратом проверки гипотез. В частности метод (тест) Дики-Фуллера

In [ ]:
p_value = adfuller(df['total'])[1]
print(
    f'Полученный уровень значимости (p-value): {round(p_value, 4)}.',
    f'{round(p_value, 4)} > 0.05. Нулевая гипотеза не отвергнута в пользу альтернативной!'
)

Как сделать ряд стационарным?

- дифференцирование ряда (простое и сезонное);
- преобразование Бокса-Кокса;
- логарифмирование ряда (частный случай преобразования Бокса-Кокса).

### Попробуем привести ряд к стационарному

In [ ]:
df.head()

In [ ]:
df['totalDiff_1'] = df['total'] - df['total'].shift(1) # Тут вроде можно использовать df['total'].diff(), но это не точно
df.head()

In [ ]:
plt.subplots(nrows=2, ncols=1, figsize=(14, 14))

ax = plt.subplot(211)
ax.set_title('Original')
ax.set_ylabel('total', rotation=0, labelpad=30)
df['total'].plot(ax=ax);

ax = plt.subplot(212)
ax.set_title('Diff')
ax.set_ylabel('totalDiff_1', rotation=0, labelpad=30)
df['totalDiff_1'].plot(ax=ax);

In [ ]:
# Проверим гипотезы для нового ряда

p_value = adfuller(df['totalDiff_1'][1:])[1]

print(
    f'Полученный уровень значимости (p-value): {round(p_value, 4)}.',
    f'{round(p_value, 4)} < 0.05. Нулевая гипотеза отвергается в пользу альтернативной!'
)

# Но на самом деле ряд не стационарный (после 2020 года дисперсия увеличилась)
# Критерий не даёт нам 100-процентную гарантию, он просто может выступать в качестве первичной проверки

In [ ]:
# Используем трюк с логарифмированием для уменьшения дисперсии и так же как и ранее составим диффериенцированный ряд
df['totalLog'] = np.log(df['total'])
df['totalLogDiff_1'] = df['totalLog'] - df['totalLog'].shift(1)
df.head()

In [ ]:
plt.subplots(nrows=2, ncols=1, figsize=(14, 14))

ax = plt.subplot(211)
ax.set_title('Original')
ax.set_ylabel('totalLog', rotation=0, labelpad=30)
df['totalLog'].plot(ax=ax);

ax = plt.subplot(212)
ax.set_title('Diff')
ax.set_ylabel('totalLogDiff_1', rotation=0, labelpad=30)
df['totalLogDiff_1'].plot(ax=ax);

In [ ]:
p_value = adfuller(df['totalLogDiff_1'][1:])[1]

print(
    f'Полученный уровень значимости (p-value): {round(p_value, 4)}.',
    f'{round(p_value, 4)} < 0.05. Нулевая гипотеза отвергается в пользу альтернативной!'
)


In [ ]:
plt.hist(df['totalLogDiff_1'], density=True, bins=40);
# Как можно видеть распредение признака 'totalLogDiff_1' не очень похоже на нормальное

In [ ]:
# Построим корелограмму и проверим наличие значимых коррелаций
plt.figure(figsize=(14, 14))

ax = plt.subplot(211)
df['totalLogDiff_1'].plot(ax=ax)

ax = plt.subplot(212)
plot_acf(df['totalLogDiff_1'][1:], lags=60, ax=ax);

Дано множество $Y_1$, $Y_2$, $Y_3$, $Y_4$, ...


А где $X$-ы?

Ответ:
* $X$-ы можно получить из метки времени, если знать периоды.
* $X$-ы можно получить из $Y$ - обобщённая авторегрессия.




In [ ]:
from statsmodels.tsa.ar_model import AutoReg

In [ ]:
train = pd.read_csv("train.csv")
train

In [ ]:
#train.fillna(method='bfill', inplace=True)
train.fillna(value=0.0, inplace=True)
train

In [ ]:
series = train.total.values

In [ ]:
ar = AutoReg(series, lags=168).fit()

In [ ]:
prediction = ar.predict(start=35064, end=35064+4344-1, dynamic=False)

In [ ]:
sample = pd.read_csv("sample.csv")
sample

In [ ]:
sample["total"] = prediction
sample

In [ ]:
sample.to_csv("prediction.csv", index=False)

## ARMA

Использовать обыкновенную регрессию для предсказания значений временного ряда плохая идея.

Попробуем решать задачу регрессии на основе истории самого ряда:

### $$y_t = \alpha + \theta_1 y_{t-1} + \theta_2 y_{t-2}, \cdots, \theta_{p} y_{t-p} + \epsilon_t$$
где $\alpha, \theta_i$ &#151; параметры модели

Такая модель для предсказания значений $y_t$ называется моделью авторегрессии с гиперпараметром $p$, **AR(p)**.




Ещё одна идея:

Попробуем решать задачу регрессии на основе истории остатков ряда:

### $$y_t = \alpha + \gamma_1 \epsilon_{t-1} + \gamma_2 \epsilon_{t-2}, \cdots, \gamma_{q} \epsilon_{t - q} + \epsilon_t$$
где $\alpha, \gamma_i$ &#151; параметры модели

Такая модель для предсказания значений $y_t$ называется моделью скользящего среднего с гиперпараметром $q$, **MA(q)**.

Если же мы представим $y_t$ в следующем виде:

### $$y_t = \alpha + \theta_1 y_{t-1} + \theta_2 y_{t-2}, \cdots, \theta_{p} y_{t-p} + \gamma_1 \epsilon_{t-1} + \gamma_2 \epsilon_{t-2}, \cdots, \gamma_{q} \epsilon_{t-q} + \epsilon_t$$

где $\alpha, \theta_i, \gamma_i$ &#151; параметры модели, то мы получим модель под названием **ARMA(p, q)** с гиперпараметрами $p$ и $q$.

## ARIMA

Основные идеи, которые лежат в основе модели ARIMA:

- если мы имеем стационарный ряд, то мы можем использовать модель ARMA(p, q);
- если ряд нестационарен, то ряд его производных (сезонных или обычных) может оказаться стационарным.

### ARIMA(p, d, q) &#151; ARMA(p, q), которая применяется к исходному ряду, продиффериенцированному $d$ раз.

## SARMA

Пусть исходный ряд имеет сезонность длины $S$.

Добавим к модели **ARMA(p, q)** $P$ сезонных авторегресионных компонент и $Q$ сезонных компонент скользащего среднего

тогда формула для $y_t$ будет выглядеть следующим образом:

### $$y_t = \alpha + \theta_1 y_{t-1} + \theta_2 y_{t-2}, \cdots, \theta_{p} y_{t-p} + \\ \theta_S y_{t-S} + \theta_{2S} y_{t-2S}, \cdots, \theta_{PS} y_{t-PS} + \\ \gamma_1 \epsilon_{t-1} + \gamma_2 \epsilon_{t-2}, \cdots, \gamma_{q} \epsilon_{t-q} + \\ \gamma_S \epsilon_{t-S} + \gamma_{2S} \epsilon_{t-2S}, \cdots, \gamma_{QS} \epsilon_{t-QS} + \epsilon_t$$

где $\alpha, \theta_i, \gamma_i$ &#151; параметры модели

Такая модель для предсказания значений $y_t$ является моделью **SARMA(p, q, P, Q)**.

## SARIMA (последнее обобщение)

Пусть исходный ряд также имеет сезонность длины $S$.

Добавим к модели **ARIMA(p, d, q)** $P$ сезонных авторегресионных компонент, $Q$ сезонных компонент скользащего среднего и применим сезонное дифференцирование $D$ раз.

Такая модель для предсказания значений $y_t$ является моделью **SARIMA(p, d, q, P, D, Q)**.

### **SARIMA(p, d, q, P, D, Q)** &#151; **SARMA(p, q, P, Q)**, которая применяется к исходному ряду, продиффериенцированному $D + d$ раз.



## Подбор параметров для алгоритмов

###  1. Параметры $\alpha$, $\theta_i$, $\gamma_i$

- данные параметры подбираются при помощи метода наименьших квадратов (МНК);
- особый интерес представляют параметры $\gamma_i$, так как $\epsilon_t$ не может быть пронаблюдаем нами (это просто шум);
- для этого параметры $\gamma_i$ оцениваются с помощью анализа остатков, который был произведён после обучении модели **AR(p, P)**;
- если шум белый, то МНК даёт оценки максимального правдоподобия (это круто).

### 2. Параметры d, D

- подобрать так, чтобы ряд стал стационарным (брутфорс?);
- стоит начинать именно с сезонного дифференцирования (параметра $D$);
- чем меньше мы в итоге продифференцируем, тем лучше будет итоговая модель.

### 3. Параметры q, Q, p, P

- нельзя подбирать эти параметры только на основе оценки правдоподобия (почему?);
- обычно для их выбора используют критерий Акаике: $AIC = -2\log{L} + 2 \cdot(P + Q + p + q + 1)$;
- начальные значения можно выбрать используя автокорреляцию;
- после этого параметры q, Q, p, P подбираются простым перебором.

- $Q \cdot S$ &#151; номер последнего сезонного лага, для которого автокорреляция значима;
- $q$ &#151; номер последнего несезонного лага, для которого автокорреляция значима;
- $P \cdot S$ &#151; номер последнего сезонного лага, для которого частичная автокорреляция значима;
- $p$ &#151; номер последнего несезонного лага, для которого частичная автокорреляция значима.


In [ ]:
import statsmodels.api as sm
p = 1
d = 1
q = 1

P = 1
D = 1
Q = 1
model = sm.tsa.statespace.SARIMAX(df['total'], order=(p, d, q), seasonal_order=(P, D, Q, 12)).fit(disp=-1)
print(model.summary())

## Анализ остатков

**Остатки** &#151; разность между фактическим значением и прогнозируемым.

### $$\hat{\epsilon_t} = y_t - \hat{y_t}$$

Предположения, которым должны подчиняться остатки:

- несмещенность (равенство среднего значения остатков нулю). Можно проверить как визуально, так и с помощью двустороннего критерия Стьюдента или Уилкоксона. Если не выполняется, то можно вручную сдвинуть значения модели на среднее от остатков;
- стационарность (отсутствие зависимости от времени). Можно проверить как визуально, так и с критерия Дики-Фуллера. Если не выполняется, то точность модели зависит от времени. Необходим визуальный анализ;
- неавтокоррелируемость (отсутствие зависимости от предыдущих остатков).

In [23]:
import numpy as np
import pandas as pd
from statsmodels.tsa.ar_model import AutoReg

# 1. Загрузка данных соревнования
train = pd.read_csv('train.csv')
sample = pd.read_csv('sample.csv')

# 2. Заменяем пропуски нулями и берём почасовой ряд потребления
train.replace(np.nan, 0, inplace=True)
series = train['total'].values

# 3. Обучаем авторегрессию с недельной сезонностью (168 часов)
ar = AutoReg(series, lags=168).fit()

# 4. Прогноз на первую половину 2009 года: 4344 почасовых значения
n_train = len(series)
n_forecast = len(sample)
prediction = ar.predict(start=n_train, end=n_train + n_forecast - 1, dynamic=False)

# 5. Записываем прогноз в колонку total файла sample и сохраняем в нужном формате
sample['total'] = prediction
sample.to_csv('prediction.csv', index=False)
print(sample.head(13))
print('Строк в submission:', len(sample))

# 6. Скачиваем готовый submission на компьютер
from google.colab import files
files.download('prediction.csv')

               datetime          total
0   01.01.2009 00:00:00  141795.160409
1   01.01.2009 01:00:00  122348.849896
2   01.01.2009 02:00:00  109032.275479
3   01.01.2009 03:00:00  101638.328050
4   01.01.2009 04:00:00  102951.070940
5   01.01.2009 05:00:00  119061.216333
6   01.01.2009 06:00:00  154758.248437
7   01.01.2009 07:00:00  184614.100027
8   01.01.2009 08:00:00  194278.061611
9   01.01.2009 09:00:00  199814.505218
10  01.01.2009 10:00:00  202669.896611
11  01.01.2009 11:00:00  202086.472466
12  01.01.2009 12:00:00  198270.912058
Строк в submission: 4344


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>